In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np

from optimal_long_short.model_params import KouParams
from optimal_long_short.market_params import MarketParams
from optimal_long_short.strategy import UnitExposureLongShortStrategy
from optimal_long_short.inversion import TalbotInverter
from optimal_long_short.moments import ConditionalMoments
from optimal_long_short.monte_carlo import MonteCarlo

In [2]:
# --- shared parameters ---
# eta_{i,±} are the **means** of positive/negative jump sizes (Sepp 2004 notation)
params = KouParams(
    mu1=0.05, sigma1=0.20, lam1=1.0, p1=0.5, eta1_pos=0.10, eta1_neg=0.125,
    mu2=0.03, sigma2=0.15, lam2=0.8, p2=0.5, eta2_pos=1/12, eta2_neg=1/9,
    rho=0.95,
)
market = MarketParams(b=0.8, S10=1.0, S20=1.0)

h0 = 100
T  = 1.0
strategy = UnitExposureLongShortStrategy(h0=h0, market=market, T=T)

print(f"h0={h0},  T={T},  w1={strategy.w1:.4f},  w2={strategy.w2:.4f}")
print(f"jump_compensator1 = {params.jump_compensator1:.6f}")
print(f"jump_compensator2 = {params.jump_compensator2:.6f}")
print(f"effective_mu1     = {params.effective_mu1:.6f}")
print(f"effective_mu2     = {params.effective_mu2:.6f}")

h0=100,  T=1.0,  w1=1.0000,  w2=0.0000
jump_compensator1 = 0.020000
jump_compensator2 = 0.007614
effective_mu1     = 0.030000
effective_mu2     = 0.022386


In [3]:
# --- Laplace inversion (semi-analytical) ---
cm = ConditionalMoments(params=params, strategy=strategy, inverter=TalbotInverter(M=36))

lap_p_surv = cm.p_surv()
lap_mean   = cm.conditional_mean()
lap_var    = cm.conditional_variance()

print("Laplace inversion")
print(f"  p_surv             = {lap_p_surv:.6f}")
print(f"  E[Pi | surv]       = {lap_mean:.6f}")
print(f"  Var(Pi | surv)     = {lap_var:.6f}")
print(f"  Std(Pi | surv)     = {np.sqrt(lap_var):.6f}")

Laplace inversion
  p_surv             = 1.000000
  E[Pi | surv]       = 1.051271
  Var(Pi | surv)     = 0.074222
  Std(Pi | surv)     = 0.272438


In [4]:
# --- Monte Carlo ---
mc = MonteCarlo(params=params, strategy=strategy, n_paths=100_000, n_steps=252, seed=42)
result = mc.run()

mc_p_surv = result.p_surv
mc_mean   = result.conditional_mean
mc_var    = result.conditional_variance

# standard errors
n_surv = result.survived.sum()
se_mean = np.std(result.payoff[result.survived]) / np.sqrt(n_surv)
se_p    = np.sqrt(mc_p_surv * (1 - mc_p_surv) / len(result.survived))

print("Monte Carlo  (n_paths=100,000, n_steps=252)")
print(f"  p_surv             = {mc_p_surv:.6f}  (±{2*se_p:.6f})")
print(f"  E[Pi | surv]       = {mc_mean:.6f}  (±{2*se_mean:.6f})")
print(f"  Var(Pi | surv)     = {mc_var:.6f}")
print(f"  Std(Pi | surv)     = {np.sqrt(mc_var):.6f}")

Monte Carlo  (n_paths=100,000, n_steps=252)
  p_surv             = 1.000000  (±0.000000)
  E[Pi | surv]       = 1.052133  (±0.001722)
  Var(Pi | surv)     = 0.074164
  Std(Pi | surv)     = 0.272330


In [5]:
# --- comparison table ---
header = f"{'Metric':<22} {'Laplace':>12} {'Monte Carlo':>14} {'Abs diff':>12}"
print(header)
print("-" * len(header))

rows = [
    ("p_surv",         lap_p_surv, mc_p_surv),
    ("E[Pi | surv]",   lap_mean,   mc_mean),
    ("Var(Pi | surv)", lap_var,    mc_var),
    ("Std(Pi | surv)", np.sqrt(lap_var), np.sqrt(mc_var)),
]
for label, lap, mc_val in rows:
    print(f"{label:<22} {lap:>12.6f} {mc_val:>14.6f} {abs(lap - mc_val):>12.6f}")

Metric                      Laplace    Monte Carlo     Abs diff
---------------------------------------------------------------
p_surv                     1.000000       1.000000     0.000000
E[Pi | surv]               1.051271       1.052133     0.000862
Var(Pi | surv)             0.074222       0.074164     0.000059
Std(Pi | surv)             0.272438       0.272330     0.000108


In [9]:
# --- sweep over h0 ---
h0_grid = np.linspace(0.1, 1.0, 9)
inv = TalbotInverter(M=32)

print(f"{'h0':>5} | {'Lap p_surv':>10} {'MC p_surv':>10} | {'Lap mean':>10} {'MC mean':>10} | {'Lap var':>10} {'MC var':>10}")
print("-" * 80)

for h0_val in h0_grid:
    strat = UnitExposureLongShortStrategy(h0=float(h0_val), market=market, T=T)

    # Laplace
    cm_i = ConditionalMoments(params=params, strategy=strat, inverter=inv)
    l_ps  = cm_i.p_surv()
    l_mu  = cm_i.conditional_mean()
    l_var = cm_i.conditional_variance()

    # Monte Carlo
    mc_i = MonteCarlo(params=params, strategy=strat, n_paths=50_000, n_steps=252, seed=0)
    r    = mc_i.run()

    print(
        f"{h0_val:>5.2f} | {l_ps:>10.5f} {r.p_surv:>10.5f} | "
        f"{l_mu:>10.5f} {r.conditional_mean:>10.5f} | "
        f"{l_var:>10.5f} {r.conditional_variance:>10.5f}"
    )

   h0 | Lap p_surv  MC p_surv |   Lap mean    MC mean |    Lap var     MC var
--------------------------------------------------------------------------------
 0.10 |    0.62126    0.62598 |    1.47604    1.47337 |    0.47385    0.47545
 0.21 |    0.83054    0.83508 |    1.24397    1.24187 |    0.30946    0.31147
 0.33 |    0.91829    0.92114 |    1.15376    1.15323 |    0.23833    0.23946
 0.44 |    0.96065    0.96348 |    1.10954    1.10912 |    0.19852    0.19954
 0.55 |    0.98114    0.98242 |    1.08668    1.08726 |    0.17218    0.17279
 0.66 |    0.99100    0.99130 |    1.07441    1.07574 |    0.15305    0.15326
 0.78 |    0.99573    0.99588 |    1.06753    1.06887 |    0.13848    0.13865
 0.89 |    0.99798    0.99822 |    1.06346    1.06466 |    0.12713    0.12736
 1.00 |    0.99905    0.99920 |    1.06089    1.06208 |    0.11817    0.11838
